# simpleNMR → JASON: write assignments back into a `.jjh5`

Reads the simpleNMR result file (`input_py.json`) and writes the experimental
assignments into the JASON document using the new `Molecule.set_assignments()`
API in beautifuljason (the same call used by JEOL's `auto_assignment.py`).

**How it maps things**

* simpleNMR `catoms[*].atomNumber` (1-based, molfile order) → JASON molecule atom index,
  checked against the heavy-atom bond graph (identity first, graph isomorphism as fallback).
* The JASON **predicted** 1H/13C shifts (`molecule.spectra[*].shifts`) are the objects that get assigned;
  a shift belongs to an atom when the atom index is in `shift.nums` (1H shifts are keyed on their carbon).
* The simpleNMR experimental ppm values are matched back to the actual signals in the `.jjh5`:
  * **13C** → nearest peak in a 1D 13C spectrum → `(shift, peak)`
  * **1H** → nearest HSQC cross-peak (H and C both checked) → `(shift, peak, h_dim)`;
    falls back to a 1H multiplet / 1D peak → `(shift, multiplet)`

`DRY_RUN = True` by default: run everything, read the report, then flip it to `False` and re-run the last cell.
A timestamped backup of the `.jjh5` is made before writing.

In [1]:
# ---------------------------------------------------------------- configuration
from pathlib import Path

SIMPLENMR_JSON = Path('input_py.json')   # simpleNMR output
JJH5_FILE      = None                    # None → look for input.jjh5 next to the JSON, then workingDirectory/workingFilename.jjh5
MOLECULE_INDEX = 0                       # zero-based molecule in the JASON document

DRY_RUN     = True     # True: report only.  False: write assignments into the .jjh5
MAKE_BACKUP = True     # copy the .jjh5 to <name>.<timestamp>.bak.jjh5 before writing

C13_TOL    = 0.5       # ppm: max |simpleNMR 13C − jjh5 13C peak|
H1_TOL     = 0.05      # ppm: max |simpleNMR 1H − jjh5 1H signal|
HSQC_C_TOL = 1.5       # ppm: carbon-axis window when picking the HSQC peak for a proton

C13_SPECTRUM_INDEX  = None   # force a particular doc.nmr_data index for 13C 1D (None = auto)
HSQC_SPECTRUM_INDEX = None   # force a particular doc.nmr_data index for HSQC   (None = auto)
H1_SPECTRUM_INDEX   = None   # force a particular doc.nmr_data index for 1H 1D  (None = auto)

In [2]:
# ---------------------------------------------------------------- imports
import json
import math
import shutil
from dataclasses import dataclass, field
from datetime import datetime

import pandas as pd
import beautifuljason as bjason

H1  = bjason.Molecule.Atom.NuclType.H1
C13 = bjason.Molecule.Atom.NuclType.C13
ELEMENT_H = int(bjason.Molecule.Atom.Type.H)

print('beautifuljason', getattr(bjason, '__version__', '?'),
      '| set_assignments available:', hasattr(bjason.Molecule, 'set_assignments'))

beautifuljason 1.3.0b1 | set_assignments available: True


## 1 · Read the simpleNMR JSON

In [3]:
@dataclass
class SnmrCarbon:
    json_idx: int                 # 0-based molfile index (atomNumber − 1)
    label: str                    # e.g. C7
    c_ppm: float | None           # experimental 13C from simpleNMR
    h_ppms: list = field(default_factory=list)   # experimental 1H (1 or 2 values for CH2)
    n_h: int = 0
    c_pred: float | None = None   # simpleNMR's predicted 13C (for the report only)


def _finite(v, limit):
    try:
        v = float(v)
    except (TypeError, ValueError):
        return None
    return v if math.isfinite(v) and abs(v) < limit else None   # simpleNMR uses 10000 as "no value"


def load_simplenmr(path: Path):
    data = json.loads(Path(path).read_text())
    atoms = data.get('catoms') or data.get('nodes_now')
    if not atoms:
        raise ValueError('No "catoms" / "nodes_now" in the simpleNMR JSON')
    carbons = []
    for a in atoms:
        if a.get('symbol') != 'C':
            continue
        h = [x for x in (_finite(v, 100) for v in a.get('H1_ppm', []) or []) if x is not None]
        carbons.append(SnmrCarbon(
            json_idx=int(a['atomNumber']) - 1,
            label=f"C{a['atomNumber']}",
            c_ppm=_finite(a.get('ppm'), 1000),
            h_ppms=sorted(h),
            n_h=int(a.get('numProtons', 0) or 0),
            c_pred=_finite(a.get('ppm_calculated'), 1000),
        ))
    carbons.sort(key=lambda c: c.json_idx)

    # heavy-atom graph from simpleNMR (molgraph ids are 0-based molfile indices)
    mg = data['molgraph']
    json_elems = {int(n['id']): int(n['atom_number']) for n in mg['nodes']}
    json_edges = {frozenset((int(l['source']), int(l['target']))) for l in mg['links']}
    return data, carbons, json_elems, json_edges, molfile_xy(data.get('molfile', ''))


def molfile_xy(molblock: str) -> dict:
    """{0-based atom index: (x, y)} from a V2000 molfile (used only to break symmetry ties)."""
    lines = molblock.splitlines()
    for k, line in enumerate(lines):
        if line.rstrip().endswith('V2000'):
            n = int(line[:3])
            return {i: (float(lines[k + 1 + i][0:10]), float(lines[k + 1 + i][10:20])) for i in range(n)}
    return {}


def resolve_jjh5(json_path: Path, data: dict, explicit=None) -> Path:
    candidates = []
    if explicit:
        candidates.append(Path(explicit))
    candidates.append(Path(json_path).parent / 'input.jjh5')
    wd, wf = data.get('workingDirectory'), data.get('workingFilename')
    if wd and wf:
        candidates.append(Path(wd) / f'{wf}.jjh5')
    for c in candidates:
        if c.expanduser().is_file():
            return c.expanduser().resolve()
    raise FileNotFoundError('No .jjh5 found; tried:\n  ' + '\n  '.join(map(str, candidates)))


snmr_data, snmr_carbons, json_elems, json_edges, json_xy = load_simplenmr(SIMPLENMR_JSON)
jjh5_path = resolve_jjh5(SIMPLENMR_JSON, snmr_data, JJH5_FILE)
print('SMILES :', snmr_data.get('smilesString'))
print('jjh5   :', jjh5_path)
pd.DataFrame([{'atom': c.label, 'nH': c.n_h, '13C exp': c.c_ppm, '13C pred (sNMR)': c.c_pred,
               '1H exp': ', '.join(f'{h:.3f}' for h in c.h_ppms)} for c in snmr_carbons])

SMILES : CC(=O)C=CC1C(C)=CCCC1(C)C
jjh5   : /Users/vsmw51/Library/CloudStorage/OneDrive-DurhamUniversity/projects/programming/2025/python/awh/simpleNMRjeolTools2/archive/development/input.jjh5


,atom,nH,13C exp,13C pred (sNMR),1H exp
0,C1,0,198.310812,197.199997,
1,C2,1,148.924513,148.250000,6.593
2,C3,1,132.271062,132.449997,6.028
3,C4,0,131.840154,132.050003,
4,C5,1,122.594516,122.550003,5.477
5,C6,1,54.248267,54.250000,2.261
6,C7,0,32.457957,32.400002,
7,C8,2,31.172063,29.900002,"1.208, 1.439"
8,C9,3,27.739804,27.361111,0.830
9,C10,3,26.881624,27.100000,2.231


## 2 · Helpers: atom mapping, predicted shifts, experimental signals

In [4]:
def jason_heavy_graph(molecule):
    elems, edges = {}, set()
    for i, atom in enumerate(molecule.atoms):
        if int(atom.type) != ELEMENT_H:
            elems[i] = int(atom.type)
    for i in elems:
        for j in molecule.atoms[i].bonded_atom_numbers:
            j = int(j)
            if j in elems:
                edges.add(frozenset((i, j)))
    return elems, edges


def _procrustes_rmsd(a, b):
    """RMSD after optimal rotation/reflection + scaling of 2D point set b onto a."""
    import numpy as np
    a = np.asarray(a, float); b = np.asarray(b, float)
    a = a - a.mean(0); b = b - b.mean(0)
    na, nb = np.linalg.norm(a), np.linalg.norm(b)
    if na == 0 or nb == 0:
        return 0.0
    a, b = a / na, b / nb
    u, _, vt = np.linalg.svd(b.T @ a)
    return float(np.sqrt(((b @ (u @ vt) - a) ** 2).sum() / len(a)))


def map_atoms(json_elems, json_edges, jason_elems, jason_edges, json_xy=None, jason_xy=None):
    """Return ({json_idx: jason_idx}, note). Identity if the graphs agree, else graph isomorphism.
    Symmetry-equivalent candidates are ranked by 2D-coordinate agreement (then by identity count)."""
    if json_elems == jason_elems and json_edges == jason_edges:
        return {i: i for i in json_elems}, 'identity (same atom order)'

    import networkx as nx
    from networkx.algorithms import isomorphism as iso
    g_json, g_jason = nx.Graph(), nx.Graph()
    for i, e in json_elems.items():
        g_json.add_node(i, el=e)
    for i, e in jason_elems.items():
        g_jason.add_node(i, el=e)
    g_json.add_edges_from(tuple(e) for e in json_edges)
    g_jason.add_edges_from(tuple(e) for e in jason_edges)
    gm = iso.GraphMatcher(g_json, g_jason, node_match=iso.categorical_node_match('el', None))
    use_xy = bool(json_xy) and bool(jason_xy)
    best, n = None, 0
    for m in gm.isomorphisms_iter():
        n += 1
        keys = sorted(m)
        rmsd = _procrustes_rmsd([json_xy[k] for k in keys], [jason_xy[m[k]] for k in keys]) if use_xy else 0.0
        score = (rmsd, -sum(1 for k, v in m.items() if k == v))
        if best is None or score < best[0]:
            best = (score, dict(m))
        if n >= 5000:
            break
    if best is None:
        raise ValueError('simpleNMR molecule and JASON molecule have different structures')
    note = f'graph isomorphism ({n}{"+" if n >= 5000 else ""} candidate mappings'
    if n > 1:
        note += (f', symmetry resolved by 2D coordinates, rmsd={best[0][0]:.3f})' if use_xy
                 else ') – topologically equivalent atoms present; check gem-dimethyls etc. in the report')
    else:
        note += ')'
    return best[1], note


def prediction_spectrum(molecule, nucleus):
    specs = [s for s in molecule.spectra if s.nucleus == nucleus]
    return max(specs, key=lambda s: len(s.shifts)) if specs else None


def shifts_by_atom(spectrum):
    """{jason_atom_idx: [(shift, predicted_value, label)]} – ignored shifts skipped."""
    out = {}
    if spectrum is None:
        return out
    prefix = 'H' if spectrum.nucleus == H1 else 'C'
    for shift in spectrum.shifts:
        if shift.ignored_auto or shift.ignored_user:
            continue
        value, _ = shift.get_value_error_pair(bjason.Molecule.CalcMethod.Best)
        nums = tuple(int(n) for n in shift.nums)
        mark = f'({shift.mark})' if shift.mark else ''
        label = '/'.join(f'{prefix}{n}{mark}' for n in nums)
        for n in nums:
            out.setdefault(n, []).append((shift, None if value is None else float(value), label))
    for lst in out.values():
        lst.sort(key=lambda t: (t[1] is None, t[1]))
    return out


def _compound_peaks(spectrum):
    try:
        peaks = list(spectrum.peaks)
    except KeyError:
        return []
    keep = []
    for p in peaks:
        try:
            ok = p.classification == bjason.NMRPeak.PeakClassification.Compound
        except (KeyError, AttributeError):
            ok = True
        if ok:
            keep.append(p)
    return keep


def _multiplets(spectrum):
    try:
        return list(spectrum.multiplets)
    except KeyError:
        return []


def _nuclides(spectrum):
    return tuple(spectrum.spec_info.nuclides)


def _nearest(items, key, target, tol):
    if target is None or not items:
        return None
    best = min(items, key=lambda it: abs(key(it) - target))
    return best if abs(key(best) - target) <= tol else None


def _pick_spectrum(document, forced, predicate, score):
    idxs = [i for i, s in enumerate(document.nmr_data) if predicate(s)]
    if forced is not None:
        return forced, document.nmr_data[forced]
    if not idxs:
        return None, None
    best = max(idxs, key=lambda i: score(document.nmr_data[i]))
    return best, document.nmr_data[best]


def find_experimental_spectra(document, carbons):
    # 13C 1D: the one matching most simpleNMR carbon shifts
    def c13_score(s):
        peaks = _compound_peaks(s)
        hits = sum(1 for c in carbons if _nearest(peaks, lambda p: float(p.pos[0]), c.c_ppm, C13_TOL))
        return (hits, len(peaks))
    c13 = _pick_spectrum(document, C13_SPECTRUM_INDEX,
                         lambda s: s.ndim == 1 and _nuclides(s) == ('13C',), c13_score)

    # HSQC: the 1H/13C 2D spectrum where most (H, C) pairs land on a cross-peak (beats HMBC)
    def hsqc_score(s):
        nuc = _nuclides(s)
        hd, cd = nuc.index('1H'), nuc.index('13C')
        peaks = _compound_peaks(s)
        hits = 0
        for c in carbons:
            for h in c.h_ppms:
                if _hsqc_peak(peaks, hd, cd, h, c.c_ppm):
                    hits += 1
        return (hits, -len(peaks))
    hsqc = _pick_spectrum(document, HSQC_SPECTRUM_INDEX,
                          lambda s: s.ndim == 2 and set(_nuclides(s)) == {'1H', '13C'}, hsqc_score)

    h1 = _pick_spectrum(document, H1_SPECTRUM_INDEX,
                        lambda s: s.ndim == 1 and _nuclides(s) == ('1H',),
                        lambda s: (len(_multiplets(s)), len(_compound_peaks(s))))
    return {'13C': c13, 'HSQC': hsqc, '1H': h1}


def _hsqc_peak(peaks, hd, cd, h_ppm, c_ppm):
    cands = [p for p in peaks
             if abs(float(p.pos[hd]) - h_ppm) <= H1_TOL
             and (c_ppm is None or abs(float(p.pos[cd]) - c_ppm) <= HSQC_C_TOL)]
    if not cands:
        return None
    return min(cands, key=lambda p: (float(p.pos[hd]) - h_ppm) ** 2 / H1_TOL ** 2
               + (0 if c_ppm is None else (float(p.pos[cd]) - c_ppm) ** 2 / HSQC_C_TOL ** 2))

## 3 · Build the `(shift, signal[, dim])` assignment tuples

In [5]:
def _obj_key(o):
    g = getattr(o, 'h5_group', None)
    return getattr(g, 'name', None) or id(o)


def _pair(shifts, signals):
    """Pair predicted shifts with experimental signals (both sorted by ppm)."""
    if not shifts or not signals:
        return []
    if len(shifts) == len(signals):
        return list(zip(shifts, signals))
    if len(shifts) == 1:                       # one shift, e.g. two diastereotopic H signals
        return [(shifts[0], s) for s in signals]
    if len(signals) == 1:                      # several shifts on one signal
        return [(sh, signals[0]) for sh in shifts]
    n = min(len(shifts), len(signals))
    return list(zip(shifts[:n], signals[:n]))


def build_assignments(document, molecule_index=MOLECULE_INDEX):
    molecule = document.mol_data[molecule_index]
    jason_elems, jason_edges = jason_heavy_graph(molecule)
    try:
        jason_xy = {i: (float(molecule.atoms[i].x), float(molecule.atoms[i].y)) for i in jason_elems}
    except (KeyError, AttributeError):
        jason_xy = {}
    mapping, map_note = map_atoms(json_elems, json_edges, jason_elems, jason_edges, json_xy, jason_xy)

    pred = {C13: shifts_by_atom(prediction_spectrum(molecule, C13)),
            H1:  shifts_by_atom(prediction_spectrum(molecule, H1))}
    spectra = find_experimental_spectra(document, snmr_carbons)
    (c13_i, c13_spec), (hsqc_i, hsqc_spec), (h1_i, h1_spec) = spectra['13C'], spectra['HSQC'], spectra['1H']

    c13_peaks = _compound_peaks(c13_spec) if c13_spec is not None else []
    if hsqc_spec is not None:
        nuc = _nuclides(hsqc_spec)
        hsqc_hd, hsqc_cd = nuc.index('1H'), nuc.index('13C')
        hsqc_peaks = _compound_peaks(hsqc_spec)
    else:
        hsqc_peaks = []
    h1_signals, h1_kind = [], None
    if h1_spec is not None:
        h1_signals, h1_kind = _multiplets(h1_spec), '1H multiplet'
        if not h1_signals:
            h1_signals, h1_kind = _compound_peaks(h1_spec), '1H 1D peak'

    tuples, rows, seen = [], [], set()

    def add(tup, row):
        key = tuple(_obj_key(o) if not isinstance(o, int) else o for o in tup)
        if key not in seen:
            seen.add(key)
            tuples.append(tup)
        rows.append(row)

    for c in snmr_carbons:
        j = mapping.get(c.json_idx)
        base = {'atom': c.label, 'jason atom': j}

        # ---------- 13C
        c_shifts = pred[C13].get(j, [])
        peak = _nearest(c13_peaks, lambda p: float(p.pos[0]), c.c_ppm, C13_TOL)
        if c.c_ppm is None:
            status = 'no 13C value in simpleNMR'
        elif not c_shifts:
            status = 'no (non-ignored) predicted 13C shift on atom'
        elif peak is None:
            status = f'no 13C peak within {C13_TOL} ppm'
        else:
            status = 'assigned'
        for sh, val, lab in (c_shifts or [(None, None, '')]):
            row = {**base, 'nucleus': '13C', 'shift': lab, 'predicted': val, 'simpleNMR': c.c_ppm,
                   'jjh5 signal': None if peak is None else float(peak.pos[0]),
                   'source': f'13C 1D [nmr_data {c13_i}]', 'status': status}
            if status == 'assigned':
                add((sh, peak), row)
            else:
                rows.append(row)

        # ---------- 1H
        if not c.h_ppms:
            continue
        h_shifts = pred[H1].get(j, [])
        signals = []           # (signal, dim, ppm, source)
        for h in c.h_ppms:
            p = _hsqc_peak(hsqc_peaks, hsqc_hd, hsqc_cd, h, c.c_ppm) if hsqc_peaks else None
            if p is not None:
                signals.append((p, hsqc_hd, float(p.pos[hsqc_hd]), f'HSQC [nmr_data {hsqc_i}]', h))
                continue
            m = _nearest(h1_signals, lambda s: float(s.pos[0]), h, H1_TOL)
            if m is not None:
                signals.append((m, None, float(m.pos[0]), f'{h1_kind} [nmr_data {h1_i}]', h))
            else:
                rows.append({**base, 'nucleus': '1H', 'shift': '', 'predicted': None, 'simpleNMR': h,
                             'jjh5 signal': None, 'source': '', 'status': f'no 1H signal within {H1_TOL} ppm'})
        if not h_shifts:
            for sig in signals:
                rows.append({**base, 'nucleus': '1H', 'shift': '', 'predicted': None, 'simpleNMR': sig[4],
                             'jjh5 signal': sig[2], 'source': sig[3],
                             'status': 'no (non-ignored) predicted 1H shift on atom'})
            continue
        pairs = _pair(h_shifts, sorted(signals, key=lambda s: s[2]))
        if len(h_shifts) != len(signals) and signals:
            pair_note = f' ({len(h_shifts)} shift(s) ↔ {len(signals)} signal(s))'
        else:
            pair_note = ''
        for (sh, val, lab), (sig, dim, ppm, src, h) in pairs:
            tup = (sh, sig) if dim is None else (sh, sig, dim)
            add(tup, {**base, 'nucleus': '1H', 'shift': lab, 'predicted': val, 'simpleNMR': h,
                      'jjh5 signal': ppm, 'source': src, 'status': 'assigned' + pair_note})

    report = pd.DataFrame(rows)
    if not report.empty:
        report['Δ (jjh5 − sNMR)'] = report['jjh5 signal'] - report['simpleNMR']
    info = {'mapping': map_note,
            '13C spectrum': c13_i, 'HSQC spectrum': hsqc_i, '1H spectrum': h1_i,
            'tuples': len(tuples)}
    return molecule, tuples, report, info

## 4 · Inspect the JASON document (read-only)

In [6]:
with bjason.Document(str(jjh5_path), mode='r') as doc:
    print(f'{len(doc.mol_data)} molecule(s); using index {MOLECULE_INDEX}')
    for i, s in enumerate(doc.nmr_data):
        print(f'  nmr_data[{i}]  {s.ndim}D  {_nuclides(s)}  peaks={len(_compound_peaks(s))}  '
              f'multiplets={len(_multiplets(s))}')
    mol = doc.mol_data[MOLECULE_INDEX]
    for sp in mol.spectra:
        print(f'  prediction {sp.nucleus.name}: {len(sp.shifts)} shifts')
    _, tuples_preview, report, info = build_assignments(doc)

for k, v in info.items():
    print(f'{k:>14}: {v}')
pd.set_option('display.float_format', '{:.4f}'.format)
report

1 molecule(s); using index 0
  nmr_data[0]  1D  ('1H',)  peaks=74  multiplets=12
  nmr_data[1]  2D  ('1H', '13C')  peaks=11  multiplets=11
  nmr_data[2]  2D  ('1H', '1H')  peaks=9  multiplets=0
  nmr_data[3]  2D  ('1H', '13C')  peaks=16  multiplets=0
  nmr_data[4]  1D  ('13C',)  peaks=13  multiplets=0
  prediction C13: 13 shifts
  prediction H1: 12 shifts
       mapping: identity (same atom order)
  13C spectrum: 4
 HSQC spectrum: 1
   1H spectrum: 0
        tuples: 25


,atom,jason atom,nucleus,shift,predicted,simpleNMR,jjh5 signal,source,status,Δ (jjh5 − sNMR)
0,C1,0,13C,C0,197.2000,198.3108,198.3108,13C 1D [nmr_data 4],assigned,0.0000
1,C2,1,13C,C1,148.2500,148.9245,148.9245,13C 1D [nmr_data 4],assigned,0.0000
2,C2,1,1H,H1,6.5303,6.5926,6.5926,HSQC [nmr_data 1],assigned,0.0000
3,C3,2,13C,C2,132.4500,132.2711,132.2711,13C 1D [nmr_data 4],assigned,0.0000
4,C3,2,1H,H2,6.0900,6.0282,6.0282,HSQC [nmr_data 1],assigned,0.0000
5,C4,3,13C,C3,132.0500,131.8402,131.8402,13C 1D [nmr_data 4],assigned,0.0000
6,C5,4,13C,C4,122.5500,122.5945,122.5945,13C 1D [nmr_data 4],assigned,0.0000
7,C5,4,1H,H4,5.4600,5.4770,5.4770,HSQC [nmr_data 1],assigned,0.0000
8,C6,5,13C,C5,54.2500,54.2483,54.2483,13C 1D [nmr_data 4],assigned,0.0000
9,C6,5,1H,H5,2.4888,2.2607,2.2607,HSQC [nmr_data 1],assigned,0.0000


In [7]:
problems = report[~report['status'].str.startswith('assigned')] if not report.empty else report
print(f"{(report['status'].str.startswith('assigned')).sum()} rows assigned, {len(problems)} need attention")
problems

25 rows assigned, 0 need attention


,atom,jason atom,nucleus,shift,predicted,simpleNMR,jjh5 signal,source,status,Δ (jjh5 − sNMR)


## 5 · Write the assignments into the `.jjh5`
Set `DRY_RUN = False` in the configuration cell, re-run it, then run this cell.
`set_assignments()` replaces the molecule's assignment set with these tuples.

In [8]:
DRY_RUN = False

In [9]:
if DRY_RUN:
    print('DRY_RUN is True – nothing written. Review the report above, then set DRY_RUN = False.')
else:
    if not hasattr(bjason.Molecule, 'set_assignments'):
        raise RuntimeError('This beautifuljason has no Molecule.set_assignments – install the JEOL build that ships with auto_assignment.py')
    if MAKE_BACKUP:
        stamp = datetime.now().strftime('%Y%m%d-%H%M%S')
        backup = jjh5_path.with_name(f'{jjh5_path.stem}.{stamp}.bak{jjh5_path.suffix}')
        shutil.copy2(jjh5_path, backup)
        print('Backup :', backup)
    # h5 objects are tied to the open file, so rebuild the tuples inside the r+ session
    with bjason.Document(str(jjh5_path), mode='r+') as doc:
        molecule, tuples, report, info = build_assignments(doc)
        molecule.set_assignments(tuples)
    print(f"Wrote {info['tuples']} assignment(s) to {jjh5_path}")

Backup : /Users/vsmw51/Library/CloudStorage/OneDrive-DurhamUniversity/projects/programming/2025/python/awh/simpleNMRjeolTools2/archive/development/input.20260929-200151.bak.jjh5
Wrote 25 assignment(s) to /Users/vsmw51/Library/CloudStorage/OneDrive-DurhamUniversity/projects/programming/2025/python/awh/simpleNMRjeolTools2/archive/development/input.jjh5
